<a href="https://colab.research.google.com/github/harezzzz/Road_sesne_AI/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# ==============================================================================
# 1. SETUP & INSTALLATION
# ==============================================================================
!pip install ultralytics albumentations -q

import os
import shutil
import yaml
import random
import cv2
import numpy as np
import albumentations as A
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os

print(os.listdir('/content/drive/MyDrive'))


['sample.zip', 'Untitled form.gform', 'jhh-uvgk-mfd – 2 Oct 2023.pdf', 'cco-dgbi-kkx – 26 Nov 2023.pdf', 'fgw-qobp-cdv – 27 Nov 2023.pdf', 'fgw-qobp-cdv – 28 Nov 2023.pdf', 'nqy-gyzp-kfs – 30 Nov 2023.pdf', 'hab-dicz-qgj – 1 Dec 2023.pdf', 'RA2211027020005.pdf', 'RA2211027020054.pdf', 'RA2211027020055.pdf', 'RA2211027020063.pdf', '1620631204 (3).pdf', '1620631204 (2).pdf', '1620631204 (1).pdf', '1620631204.pdf', 'IMG_8247.png', 'ef8dfb42-117f-4a82-b6aa-f3eb5b565787 (3).jpeg', 'ef8dfb42-117f-4a82-b6aa-f3eb5b565787 (2).jpeg', 'ef8dfb42-117f-4a82-b6aa-f3eb5b565787 (1).jpeg', 'ef8dfb42-117f-4a82-b6aa-f3eb5b565787.jpeg', 'IMG_8249.png', 'IMG_8795.png', 'IMG_8796.png', 'RA2211027020054 harish viswanath.jpg', 'RA2211027020054 HARISH VISWANATH.pptx', '_MG_1864.JPG', '_MG_1868.JPG', '_MG_1985.JPG', '_MG_1986.JPG', '_MG_1987.JPG', '_MG_1988.JPG', '_MG_2699.JPG', '_MG_2700.JPG', 'HARISH CC (1).pptx', 'Untitled.pdf', 'harish cc_report_(1) (1).pdf', 'WhatsApp Image 2024-10-26 at 13.45.12_650e1247 (

In [20]:
print(os.listdir('/content/drive/MyDrive/Utility_text'))


['Utility_text']


In [24]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'Utility' in root:
        print("Found:", root)


Found: /content/drive/MyDrive/Utility_text
Found: /content/drive/MyDrive/Utility_text/Utility_text
Found: /content/drive/MyDrive/Utility_text/Utility_text/labels
Found: /content/drive/MyDrive/Utility_text/Utility_text/images


In [28]:
DATASET_PATH = "/content/drive/MyDrive/Utility_text/Utility_text"

print("Inside dataset folder:")
print(os.listdir(DATASET_PATH))
print("Images folder:")
print(os.listdir(os.path.join(DATASET_PATH, "images")))

print("Labels folder:")
print(os.listdir(os.path.join(DATASET_PATH, "labels")))


Inside dataset folder:
['labels', 'images']
Images folder:
['Screenshot 2025-06-10 101259.png', 'Screenshot 2025-06-10 103139.png', 'Screenshot 2025-06-10 101445.png', 'Screenshot 2025-12-15 173245.png', 'Screenshot 2025-06-10 101544.png', 'Worcester_Commonwealth_Avenue_012_R1_ErUwOyXqGHZbUKp5LN4ijQ.jpg', 'Screenshot 2025-12-16 094655.png', 'Screenshot 2025-12-16 113606.png', 'Screenshot 2025-06-09 171349.png', 'Bosten_Readville_River_Street_018_R5_TArpxqRF0ar68N6nD3AONw.jpg', 'Bosten_Readville_Vallaro_Road_001_R4_GBcwNFOA59qiRVpdmPyWyA.jpg', 'Screenshot 2025-06-10 104119.png', 'Screenshot 2025-06-10 104027.png', 'Bosten_Readville_Colchester_Street_001_R1_ERRl7I4HmClDYidBkTF1xQ.jpg', 'Bosten_Readville_Clifford_Street_001_R10_rUlTTNW4UUV5n3GL4raX8g.jpg', 'Screenshot 2025-12-17 095016.png', 'Worcester_High_Street_018_R12_4pgxcdKnSNuIM8fiQlhrQg.jpg', 'Screenshot 2025-06-09 164335.png', 'Bosten_Readville_Industrial_Drive_017_R3_g6HtuWcEc6Fl2luzhrqMkA.jpg', 'Screenshot 2025-06-10 104053.png

In [29]:

# ==============================================================================
# 2. PATHS & CONFIGURATION
# ==============================================================================
# EDIT THESE PATHS
ORIGINAL_DATA_PATH = '/content/drive/MyDrive/Utility_text/Utility_text'
PROJECT_DIR = '/content/drive/MyDrive/Utility_text/Utility_text'

# LOCAL WORKSPACE (Training is faster on local disk than Google Drive)
LOCAL_ROOT = '/content/dataset_clean'
TRAIN_DIR = os.path.join(LOCAL_ROOT, 'train')
VAL_DIR = os.path.join(LOCAL_ROOT, 'val')

# LOAD CLASS NAMES
classes_txt = os.path.join(ORIGINAL_DATA_PATH, 'labels', 'classes.txt')
with open(classes_txt, 'r') as f:
    CLASS_NAMES = [line.strip() for line in f.readlines()]

In [30]:


# ==============================================================================
# 3. SPLIT DATA (THE MOST IMPORTANT STEP TO AVOID OVERFITTING)
# ==============================================================================
print("🚀 Starting Data Split (80% Train, 20% Val)...")

if os.path.exists(LOCAL_ROOT): shutil.rmtree(LOCAL_ROOT)

all_images = [f for f in os.listdir(os.path.join(ORIGINAL_DATA_PATH, 'images')) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
train_files, val_files = train_test_split(all_images, test_size=0.20, random_state=42)

def move_to_local(files, source_base, target_base):
    os.makedirs(os.path.join(target_base, 'images'), exist_ok=True)
    os.makedirs(os.path.join(target_base, 'labels'), exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(source_base, 'images', f), os.path.join(target_base, 'images', f))
        lbl = os.path.splitext(f)[0] + '.txt'
        if os.path.exists(os.path.join(source_base, 'labels', lbl)):
            shutil.copy(os.path.join(source_base, 'labels', lbl), os.path.join(target_base, 'labels', lbl))

move_to_local(train_files, ORIGINAL_DATA_PATH, TRAIN_DIR)
move_to_local(val_files, ORIGINAL_DATA_PATH, VAL_DIR)

print(f"✅ Split Complete. Train: {len(train_files)} | Val: {len(val_files)} (Original Unseen Images)")


🚀 Starting Data Split (80% Train, 20% Val)...
✅ Split Complete. Train: 82 | Val: 21 (Original Unseen Images)


In [31]:

# ==============================================================================
# 4. BALANCE ONLY THE TRAINING SET
# ==============================================================================
print("\n🚀 Balancing Training Set with Augmentation...")

# Define heavy augmentation to ensure "unique" samples
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Rotate(limit=30, p=0.7, border_mode=cv2.BORDER_CONSTANT),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Blur(blur_limit=3, p=0.2),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

TARGET_COUNT = 600 # Realistic target to avoid over-weighting tiny classes
train_img_path = os.path.join(TRAIN_DIR, 'images')
train_lbl_path = os.path.join(TRAIN_DIR, 'labels')

# Map images to classes
class_to_images = {i: [] for i in range(len(CLASS_NAMES))}
for img_f in os.listdir(train_img_path):
    lbl_f = os.path.splitext(img_f)[0] + '.txt'
    if os.path.exists(os.path.join(train_lbl_path, lbl_f)):
        with open(os.path.join(train_lbl_path, lbl_f), 'r') as f:
            for line in f:
                cid = int(float(line.split()[0]))
                # Ensure class ID is within valid range
                if 0 <= cid < len(CLASS_NAMES):
                    if img_f not in class_to_images[cid]:
                        class_to_images[cid].append(img_f)
                else:
                    print(f"   Warning: Class ID {cid} found in {lbl_f} is out of bounds (0-{len(CLASS_NAMES)-1}). Skipping.")

# Augment underrepresented classes
for cid, imgs in class_to_images.items():
    current_count = len(imgs)
    if 0 < current_count < TARGET_COUNT:
        print(f"   Augmenting {CLASS_NAMES[cid]}: {current_count} -> {TARGET_COUNT}")
        needed = TARGET_COUNT - current_count
        for i in range(needed):
            src_name = random.choice(imgs)
            img = cv2.imread(os.path.join(train_img_path, src_name))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            bboxes, labels = [], []
            with open(os.path.join(train_lbl_path, os.path.splitext(src_name)[0] + '.txt'), 'r') as f:
                for line in f:
                    p = line.split()
                    labels.append(int(float(p[0])))
                    bboxes.append([float(x) for x in p[1:]])

            try:
                aug = transform(image=img, bboxes=bboxes, class_labels=labels)
                new_name = f"aug_{cid}_{i}_{src_name}"
                cv2.imwrite(os.path.join(train_img_path, new_name), cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR))
                with open(os.path.join(train_lbl_path, os.path.splitext(new_name)[0] + '.txt'), 'w') as f:
                    for j, b in enumerate(aug['bboxes']):
                        f.write(f"{aug['class_labels'][j]} {' '.join(map(str, b))}\n")
            except: continue



🚀 Balancing Training Set with Augmentation...
   Augmenting G or Gas: 30 -> 600


/tmp/ipython-input-4231757816.py:11: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),


   Augmenting STM: 14 -> 600
   Augmenting Unknown: 38 -> 600


In [32]:


# ==============================================================================
# 5. SMART CHECKPOINT & RESUME SYSTEM
# ==============================================================================
data_yaml = {
    'path': LOCAL_ROOT,
    'train': 'train/images',
    'val': 'val/images',  # STRICTLY original unseen data
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

# Define the run name
RUN_NAME = 'YOLOv8x_Custom_Augmented'

# Logic to find the last checkpoint on Drive
last_ckpt = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'last.pt')

if os.path.exists(last_ckpt):
    print(f"🔄 Found checkpoint! Resuming training from: {last_ckpt}")
    model = YOLO(last_ckpt)
    resume_flag = True
else:
    print("🆕 No checkpoint found. Starting fresh from YOLOv8x...")
    model = YOLO('yolov8x.pt')
    resume_flag = False

# ==============================================================================
# 6. TRAINING WITH OVERFIT PROTECTIONS & EARLY STOPPING
# ==============================================================================
model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=4,             # Adjust to 8 if you get "Out of Memory"
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,        # Keep files in the same folder
    resume=resume_flag,
    save=True,
    save_period=1,        # Save to Drive every epoch (safest for Colab)

    # --- OVERFITTING PROTECTIONS ---
    patience=10,          # EARLY STOPPING: Stop if no improvement
    dropout=0.15,         # Regularization: Prevent memorization
    weight_decay=0.0005,  # Regularization: Keep weights small
    label_smoothing=0.1,  # Generalization: Don't be "too sure" of labels

    # Augmentation during training (Mosaic is on by default)
    hsv_h=0.015,
    flipud=0.0
)

# ==============================================================================
# 7. FINAL VALIDATION (THE MOMENT OF TRUTH)
# ==============================================================================
print("\n--- Final Evaluation on CLEAN ORIGINAL DATA ---")
best_model = YOLO(os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt'))
metrics = best_model.val()
print(f"\n✅ Final Honest mAP@0.5: {metrics.box.map50}")


🆕 No checkpoint found. Starting fresh from YOLOv8x...
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.15, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,